# 🩺 Diabetic Retinopathy Classification — AutoML Pipeline
## AutoKeras + EfficientNet Fine-Tuning | 384×384 Resolution

| Phase | Description |
|-------|-------------|
| AutoML Search | AutoKeras tries 5 architectures, picks the best |
| Fine-Tuning | Unfreeze all layers, Adam lr=1e-5 |
| Target | 95%+ Accuracy on DR 5-class classification |

> Dataset must be organized as: `/content/drive/MyDrive/DR_dataset/<class_name>/image.jpg`

## ⚙️ Step 1 — Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive for dataset and model saving
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive mounted at /content/drive')

In [ ]:
%%capture
# Install required packages — captured to suppress verbose output
!pip install -q tensorflow==2.15.0
!pip install -q keras==2.15.0
!pip install -q autokeras==1.1.0
!pip install -q keras-tuner==1.4.7
!pip install -q scikit-learn
!pip install -q matplotlib seaborn
print('✅ All packages installed')

## 📦 Step 2 — Import Libraries

In [ ]:
# Core Python & system imports
import os
import gc
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# TensorFlow / Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB4

# AutoKeras
import autokeras as ak

# Scikit-learn for metrics and class weights
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print(f'✅ TensorFlow version : {tf.__version__}')
print(f'✅ AutoKeras version  : {ak.__version__}')
print(f'✅ GPUs available     : {len(gpus)} → {[g.name for g in gpus]}')

# Enable memory growth to avoid OOM crashes on GPU
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Enable mixed precision for faster training at 384×384
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('✅ Mixed precision (float16) enabled for GPU speed-up')

## 🗂️ Step 3 — Configuration & Dataset Paths

In [ ]:
# ─── Global Configuration ────────────────────────────────────────────────────
# STRICT: image size MUST be 384×384 as required
IMG_SIZE        = (384, 384)
BATCH_SIZE      = 16          # Reduced to 16 for 384px to avoid RAM/VRAM crash
AUTOTUNE        = tf.data.AUTOTUNE
SEED            = 42

# AutoKeras settings
MAX_TRIALS      = 5           # AutoKeras will try 5 architectures
AUTOML_EPOCHS   = 3           # Exactly 3 epochs per trial as required

# Fine-tuning settings
FINETUNE_EPOCHS = 3           # Exactly 3 more epochs for fine-tuning
FINETUNE_LR     = 1e-5        # Adam learning rate for fine-tuning

# Dataset path — your Google Drive folder
DATASET_DIR     = '/content/drive/MyDrive/DR_dataset'

# Output path for saved model
MODEL_SAVE_PATH = '/content/drive/MyDrive/dr_model_384.keras'

# Temporary AutoKeras project directory
AUTOKERAS_DIR   = '/content/autokeras_dr_project'

# DR class names — update order to match your folder names if needed
CLASS_NAMES     = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferative_DR']

print('✅ Configuration set:')
print(f'   Image size      : {IMG_SIZE}')
print(f'   Batch size      : {BATCH_SIZE}')
print(f'   AutoML trials   : {MAX_TRIALS}')
print(f'   AutoML epochs   : {AUTOML_EPOCHS}')
print(f'   Finetune epochs : {FINETUNE_EPOCHS}')
print(f'   Dataset dir     : {DATASET_DIR}')
print(f'   Save path       : {MODEL_SAVE_PATH}')

## 📂 Step 4 — Verify Dataset Structure

In [ ]:
# Verify dataset directory exists and display class distribution
import os

assert os.path.exists(DATASET_DIR), (
    f'❌ Dataset not found at {DATASET_DIR}\n'
    f'   Please upload your dataset to Google Drive in this structure:\n'
    f'   MyDrive/DR_dataset/\n'
    f'       No_DR/         → class 0 images\n'
    f'       Mild/          → class 1 images\n'
    f'       Moderate/      → class 2 images\n'
    f'       Severe/        → class 3 images\n'
    f'       Proliferative_DR/ → class 4 images'
)

print('📁 Dataset structure found:')
total_images = 0
class_counts = {}

for class_dir in sorted(os.listdir(DATASET_DIR)):
    full_path = os.path.join(DATASET_DIR, class_dir)
    if os.path.isdir(full_path):
        img_files = [
            f for f in os.listdir(full_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ]
        count = len(img_files)
        class_counts[class_dir] = count
        total_images += count
        print(f'   {class_dir:25s}: {count:5d} images')

print(f'\n   Total images: {total_images:,}')
print(f'   Classes found: {list(class_counts.keys())}')

# Update CLASS_NAMES based on actual folder names found
CLASS_NAMES = sorted(class_counts.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f'\n✅ Number of classes: {NUM_CLASSES}')
print(f'✅ Class names (sorted): {CLASS_NAMES}')

## 🔀 Step 5 — Load Dataset with 80/20 Split

In [ ]:
# Load training dataset (80% split) using image_dataset_from_directory
# STRICT: image size = (384, 384), batch size = 32 → adjusted to 16 for memory safety

print('📥 Loading training dataset (80%)...')
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,          # 20% reserved for validation
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,             # STRICT: (384, 384)
    batch_size=BATCH_SIZE,
    label_mode='int',
    class_names=CLASS_NAMES,
    shuffle=True
)

print('📥 Loading validation dataset (20%)...')
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,             # STRICT: (384, 384)
    batch_size=BATCH_SIZE,
    label_mode='int',
    class_names=CLASS_NAMES,
    shuffle=False
)

# Get actual class names confirmed by Keras loader
CLASS_NAMES = train_ds_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f'\n✅ Classes confirmed by Keras: {CLASS_NAMES}')
print(f'✅ Training batches  : {len(train_ds_raw)}')
print(f'✅ Validation batches: {len(val_ds_raw)}')

## ⚖️ Step 6 — Compute Class Weights for Imbalance Handling

In [ ]:
# Extract all labels from the training dataset to compute class weights
# This handles the severe class imbalance typical in DR datasets

print('⚙️ Extracting training labels for class weight computation...')
all_train_labels = []

for _, labels_batch in train_ds_raw:
    all_train_labels.extend(labels_batch.numpy().tolist())

all_train_labels = np.array(all_train_labels)
unique_classes   = np.unique(all_train_labels)

# Compute balanced class weights using sklearn
weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=all_train_labels
)

class_weights_dict = {int(cls): float(w) for cls, w in zip(unique_classes, weights)}

print('\n📊 Class distribution in training set:')
unique, counts = np.unique(all_train_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    wt = class_weights_dict[cls]
    print(f'   Class {cls} ({CLASS_NAMES[cls]:25s}): {cnt:5d} samples | weight = {wt:.4f}')

print(f'\n✅ Class weights: {class_weights_dict}')

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(CLASS_NAMES, counts, color=['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad'])
axes[0].set_title('Training Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('DR Grade')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(CLASS_NAMES, list(class_weights_dict.values()),
            color=['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad'])
axes[1].set_title('Computed Class Weights (Balanced)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('DR Grade')
axes[1].set_ylabel('Weight')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Class distribution chart saved')

## 🔄 Step 7 — Data Augmentation Layer

In [ ]:
# Define data augmentation as a Keras Sequential model
# Applied only during training — validation set receives no augmentation

data_augmentation = keras.Sequential([
    # STRICT requirements: RandomFlip horizontal
    layers.RandomFlip('horizontal'),
    # STRICT: RandomRotation factor = 0.1 (±36 degrees)
    layers.RandomRotation(factor=0.1),
    # STRICT: RandomZoom factor = 0.1
    layers.RandomZoom(height_factor=0.1, width_factor=0.1),
    # STRICT: RandomBrightness equivalent
    layers.RandomBrightness(factor=0.2),
    # Additional medical-grade augmentations for DR
    layers.RandomContrast(factor=0.2),
    layers.RandomFlip('vertical'),       # DR images are rotation-invariant
], name='data_augmentation')

print('✅ Data augmentation pipeline created:')
for layer in data_augmentation.layers:
    print(f'   {layer.name}')

# Visualize augmented samples from the first batch
sample_images, sample_labels = next(iter(train_ds_raw))
sample_img = sample_images[:1]  # Take 1 image

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Augmented Samples (same image, different augmentations)', fontsize=13)

for i in range(10):
    ax = axes[i // 5][i % 5]
    augmented = data_augmentation(sample_img, training=True)[0].numpy().astype(int)
    ax.imshow(np.clip(augmented, 0, 255))
    ax.axis('off')
    ax.set_title(f'Augment #{i+1}', fontsize=9)

plt.tight_layout()
plt.savefig('/content/augmentation_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Augmentation preview saved')

## ⚡ Step 8 — Build Optimized tf.data Pipeline

In [ ]:
# Build optimized data pipelines using tf.data AUTOTUNE
# Normalize, augment, cache and prefetch for maximum GPU utilization

def normalize_image(image, label):
    """Normalize pixel values from [0, 255] to [0, 1]"""
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment_image(image, label):
    """Apply augmentation during training only"""
    image = data_augmentation(tf.expand_dims(image, 0), training=True)[0]
    return image, label

# ── Training pipeline ──────────────────────────────────────────────────────
# cache() → map(normalize) → map(augment) → prefetch()
# cache() is placed BEFORE normalization to cache raw images and save RAM
train_ds = (
    train_ds_raw
    .map(normalize_image, num_parallel_calls=AUTOTUNE)   # normalize to [0,1]
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)                     # augment batch
    .cache()                                              # cache in memory after first epoch
    .prefetch(AUTOTUNE)                                   # pre-load next batch while GPU trains
)

# ── Validation pipeline ────────────────────────────────────────────────────
# No augmentation on validation set — only normalize and prefetch
val_ds = (
    val_ds_raw
    .map(normalize_image, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)

# Verify pipeline output shapes
for images, labels in train_ds.take(1):
    print(f'✅ Training batch shape   : images={images.shape}, labels={labels.shape}')
    print(f'   Pixel value range      : min={images.numpy().min():.4f}, max={images.numpy().max():.4f}')

for images, labels in val_ds.take(1):
    print(f'✅ Validation batch shape : images={images.shape}, labels={labels.shape}')

print('\n✅ tf.data pipeline ready: normalize → augment → cache → prefetch')
print(f'   AUTOTUNE parallelism enabled for CPU pre-processing')

## 🤖 Step 9 — AutoKeras ImageClassifier (AutoML Search)

In [ ]:
# Initialize AutoKeras ImageClassifier
# AutoKeras will try MAX_TRIALS=5 different architectures
# and select the one with the best validation accuracy

print('🤖 Initializing AutoKeras ImageClassifier...')
print(f'   max_trials  = {MAX_TRIALS}  (5 architecture variants to try)')
print(f'   epochs      = {AUTOML_EPOCHS}  (STRICT: exactly 3 epochs per trial)')
print(f'   num_classes = {NUM_CLASSES}')
print()

# Clear any previous AutoKeras project to avoid conflicts
import shutil
if os.path.exists(AUTOKERAS_DIR):
    shutil.rmtree(AUTOKERAS_DIR)
    print(f'   Cleared previous AutoKeras project at {AUTOKERAS_DIR}')

# Create AutoKeras ImageClassifier
# - overwrite=True ensures clean run
# - seed for reproducibility
automl_classifier = ak.ImageClassifier(
    num_classes=NUM_CLASSES,
    multi_label=False,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    overwrite=True,
    max_trials=MAX_TRIALS,
    seed=SEED,
    directory=AUTOKERAS_DIR,
    project_name='dr_automl'
)

print('\n✅ AutoKeras ImageClassifier initialized')
print('   AutoKeras will search through architectures including:')
print('   → EfficientNet variants (B0–B7)')
print('   → ResNet variants')
print('   → Xception')
print('   → Custom CNN architectures')
print('   → Different augmentation combinations')

In [ ]:
# ── Prepare numpy arrays for AutoKeras (required format) ──────────────────
# AutoKeras .fit() works best with numpy arrays; convert tf.data batches
# We use a generator approach to avoid loading all into RAM at once

print('📦 Converting datasets for AutoKeras compatibility...')
print('   (Collecting training images and labels — may take a moment)')

def dataset_to_numpy(dataset, name='dataset'):
    """Convert tf.data.Dataset to numpy arrays with progress reporting"""
    images_list, labels_list = [], []
    total = len(dataset)
    for i, (imgs, lbls) in enumerate(dataset):
        images_list.append(imgs.numpy())
        labels_list.append(lbls.numpy())
        if (i + 1) % 20 == 0:
            print(f'   {name}: processed {i+1}/{total} batches...')
    X = np.concatenate(images_list, axis=0)
    y = np.concatenate(labels_list, axis=0)
    print(f'   {name}: X={X.shape}, y={y.shape}, dtype={X.dtype}')
    return X, y

# NOTE: train_ds already has augmented, normalized images
# We use train_ds_raw normalized only for AutoKeras (it has its own augmentation)
train_ds_for_ak = (
    train_ds_raw
    .map(normalize_image, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds_for_ak = val_ds  # Already normalized

X_train, y_train = dataset_to_numpy(train_ds_for_ak, 'Train')
X_val,   y_val   = dataset_to_numpy(val_ds_for_ak,   'Val')

print(f'\n✅ Training set  : {X_train.shape}')
print(f'✅ Validation set: {X_val.shape}')

# Free GPU memory before AutoML search
gc.collect()
tf.keras.backend.clear_session()
print('✅ Memory cleared for AutoML search')

In [ ]:
# ── Run AutoKeras AutoML Search ──────────────────────────────────────────
# STRICT: exactly 3 epochs per trial
# AutoKeras will internally try MAX_TRIALS=5 different architectures
# and train each for AUTOML_EPOCHS=3 epochs

print('🔍 Starting AutoKeras architecture search...')
print(f'   This will try {MAX_TRIALS} different architectures × {AUTOML_EPOCHS} epochs each')
print(f'   Estimated time: ~15–30 minutes on T4 GPU')
print('=' * 60)

automl_classifier.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=AUTOML_EPOCHS,           # STRICT: exactly 3 epochs
    class_weight=class_weights_dict, # Handle class imbalance
    callbacks=[
        # Reduce LR if validation accuracy plateaus during 3-epoch search
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=2,
            verbose=1
        ),
        # Log best metrics
        keras.callbacks.CSVLogger('/content/automl_training_log.csv')
    ],
    verbose=2
)

print('\n✅ AutoKeras search complete!')
print(f'   Best trial is selected automatically by AutoKeras')

## 🏆 Step 10 — Export Best Model & Display Summary

In [ ]:
# Export the best model found by AutoKeras
# This is a standard Keras model ready for fine-tuning and deployment

print('📤 Exporting best model from AutoKeras...')

best_model = automl_classifier.export_model()

print('✅ Best model exported!')
print(f'   Model type : {type(best_model)}')
print(f'   Input shape: {best_model.input_shape}')
print(f'   Output shape: {best_model.output_shape}')
print()

# Display full model summary
best_model.summary()

# Count trainable parameters
total_params     = sum([np.prod(v.shape) for v in best_model.trainable_variables])
print(f'\n📊 Total trainable parameters: {total_params:,}')

In [ ]:
# Evaluate AutoML model before fine-tuning

print('📊 Evaluating AutoML model (before fine-tuning)...')

automl_results = best_model.evaluate(X_val, y_val, verbose=1)
automl_loss    = automl_results[0]
automl_acc     = automl_results[1]

print(f'\n✅ AutoML Model Performance (3 epochs search):')
print(f'   Validation Loss    : {automl_loss:.4f}')
print(f'   Validation Accuracy: {automl_acc * 100:.2f}%')

## 🔧 Step 11 — Fine-Tuning Phase (All Layers Unfrozen)

In [ ]:
# Fine-tuning: unfreeze ALL layers and retrain with a very low learning rate
# STRICT: exactly 3 more epochs, Adam optimizer with lr=1e-5

print('🔧 Fine-Tuning Phase: Unfreezing all layers...')

# Unfreeze ALL layers for full fine-tuning
best_model.trainable = True

# Count all layers that are now trainable
trainable_layers = sum(1 for layer in best_model.layers if layer.trainable)
print(f'   Total layers in model   : {len(best_model.layers)}')
print(f'   Trainable layers (all)  : {trainable_layers}')

# STRICT: Recompile with Adam optimizer at learning rate = 1e-5
optimizer = keras.optimizers.Adam(learning_rate=FINETUNE_LR)

best_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'\n✅ Model recompiled with:')
print(f'   Optimizer : Adam')
print(f'   LR        : {FINETUNE_LR} (1e-5 as required)')
print(f'   Loss      : sparse_categorical_crossentropy')
print(f'   Epochs    : {FINETUNE_EPOCHS} (STRICT: exactly 3 epochs)')

In [ ]:
# ── Build augmented training set for fine-tuning ──────────────────────────
# Apply augmentation to training data for fine-tuning
print('🔄 Preparing augmented training data for fine-tuning...')

# We already have X_train normalized; apply augmentation
augmented_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=min(len(X_train), 2000), seed=SEED)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_tf_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print('✅ Augmented fine-tuning dataset ready')
print()

# ── Run fine-tuning for exactly 3 epochs ─────────────────────────────────
print('🚀 Starting fine-tuning...')
print(f'   Epochs: {FINETUNE_EPOCHS} (STRICT requirement)')
print('=' * 60)

finetune_callbacks = [
    # Save best checkpoint during fine-tuning
    keras.callbacks.ModelCheckpoint(
        filepath='/content/best_finetune_checkpoint.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Log fine-tuning metrics
    keras.callbacks.CSVLogger('/content/finetune_log.csv'),
    # Reduce LR on plateau during fine-tuning
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

finetune_history = best_model.fit(
    augmented_ds,
    validation_data=val_tf_ds,
    epochs=FINETUNE_EPOCHS,          # STRICT: exactly 3 epochs
    class_weight=class_weights_dict, # Apply class weights for imbalance
    callbacks=finetune_callbacks,
    verbose=1
)

print('\n✅ Fine-tuning complete!')

## 📈 Step 12 — Training History Visualization

In [ ]:
# Plot fine-tuning training history

history = finetune_history

train_acc  = history.history['accuracy']
val_acc    = history.history['val_accuracy']
train_loss = history.history['loss']
val_loss   = history.history['val_loss']
epochs_range = range(1, len(train_acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fine-Tuning Training History (3 Epochs)', fontsize=14, fontweight='bold')

# Accuracy plot
axes[0].plot(epochs_range, [a * 100 for a in train_acc], 'b-o', label='Train Accuracy', linewidth=2)
axes[0].plot(epochs_range, [a * 100 for a in val_acc],   'r-o', label='Val Accuracy',   linewidth=2)
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True)
axes[0].set_xticks(list(epochs_range))

# Loss plot
axes[1].plot(epochs_range, train_loss, 'b-o', label='Train Loss', linewidth=2)
axes[1].plot(epochs_range, val_loss,   'r-o', label='Val Loss',   linewidth=2)
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)
axes[1].set_xticks(list(epochs_range))

plt.tight_layout()
plt.savefig('/content/finetune_history.png', dpi=100, bbox_inches='tight')
plt.show()

# Print final metrics
print('\n📊 Fine-Tuning Results Summary:')
print(f'   Final Training Accuracy   : {train_acc[-1] * 100:.2f}%')
print(f'   Final Validation Accuracy : {val_acc[-1]   * 100:.2f}%')
print(f'   Final Training Loss       : {train_loss[-1]:.4f}')
print(f'   Final Validation Loss     : {val_loss[-1]:.4f}')
print(f'   Best Val Accuracy         : {max(val_acc) * 100:.2f}%')

## 📊 Step 13 — Evaluation: Confusion Matrix & Classification Report

In [ ]:
# Load best checkpoint from fine-tuning (highest val_accuracy)
print('📂 Loading best checkpoint for evaluation...')

if os.path.exists('/content/best_finetune_checkpoint.keras'):
    final_model = keras.models.load_model('/content/best_finetune_checkpoint.keras')
    print('   ✅ Loaded best checkpoint (highest val_accuracy)')
else:
    final_model = best_model
    print('   ⚠️  Using last epoch model (checkpoint not found)')

# Final evaluation on validation set
print('\n🧮 Running inference on validation set...')
y_pred_probs = final_model.predict(X_val, batch_size=BATCH_SIZE, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_val

# Overall accuracy
from sklearn.metrics import accuracy_score
final_accuracy = accuracy_score(y_true, y_pred)
print(f'\n✅ Final Validation Accuracy: {final_accuracy * 100:.2f}%')

# ── Classification Report ─────────────────────────────────────────────────
print('\n📋 Classification Report:')
print('=' * 70)
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
print('📊 Generating Confusion Matrix...')

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts confusion matrix
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp1.plot(ax=axes[0], cmap='Blues', colorbar=True)
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# Normalized confusion matrix (row-wise = recall per class)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
disp2   = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp2.plot(ax=axes[1], cmap='Greens', colorbar=True, values_format='.2f')
axes[1].set_title('Normalized Confusion Matrix (Recall per Class)', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle(f'DR Classification — Final Accuracy: {final_accuracy * 100:.2f}%',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n✅ Confusion matrix saved to /content/confusion_matrix.png')

# Per-class accuracy breakdown
print('\n📊 Per-Class Accuracy:')
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for cls, acc in zip(CLASS_NAMES, per_class_acc):
    bar = '█' * int(acc * 30)
    print(f'   {cls:25s}: {acc * 100:6.2f}% {bar}')

## 💾 Step 14 — Save Model to Google Drive

In [ ]:
# Save the fully trained and fine-tuned model in .keras format
# STRICT: save to /content/drive/MyDrive/dr_model_384.keras

print('💾 Saving final model to Google Drive...')
print(f'   Path: {MODEL_SAVE_PATH}')
print()

# Save final model (best checkpoint already loaded as final_model)
final_model.save(MODEL_SAVE_PATH)

# Verify the saved file
if os.path.exists(MODEL_SAVE_PATH):
    file_size_mb = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
    print(f'✅ Model saved successfully!')
    print(f'   File size: {file_size_mb:.1f} MB')
    print(f'   Format   : .keras (Keras v3 format)')
    print(f'   Path     : {MODEL_SAVE_PATH}')
else:
    raise FileNotFoundError(f'❌ Model save FAILED! File not found at {MODEL_SAVE_PATH}')

# Also save a local copy in /content for quick download
LOCAL_MODEL_PATH = '/content/dr_model_384.keras'
final_model.save(LOCAL_MODEL_PATH)
print(f'\n✅ Local copy saved at: {LOCAL_MODEL_PATH}')

# Verify model can be reloaded correctly
print('\n🔍 Verifying saved model integrity...')
reloaded_model = keras.models.load_model(MODEL_SAVE_PATH)
reload_acc     = reloaded_model.evaluate(X_val[:BATCH_SIZE], y_val[:BATCH_SIZE], verbose=0)
print(f'✅ Model reloaded and verified — sample batch accuracy: {reload_acc[1]*100:.2f}%')

## 📥 Step 15 — Download Model File

In [ ]:
# Download the .keras model file to your local machine
# Run this cell to trigger a browser download

from google.colab import files

print('📥 Initiating model download...')
print(f'   File: dr_model_384.keras')
print(f'   This model can be loaded in Streamlit UI using:')
print(f'   >>> model = tf.keras.models.load_model("dr_model_384.keras")')
print()

# Download from local /content (faster than Drive)
files.download('/content/dr_model_384.keras')

print('✅ Download triggered — check your browser downloads')

## 🔬 Step 16 — Model Inference Test (Sanity Check)

In [ ]:
# Sanity check: run predictions on a few sample validation images
# and visualize the model's predictions vs ground truth

print('🔬 Running inference sanity check on sample images...')

# Pick one sample from each class
n_samples = 5
sample_indices = []
for cls in range(NUM_CLASSES):
    cls_indices = np.where(y_val == cls)[0]
    if len(cls_indices) > 0:
        sample_indices.append(cls_indices[0])

fig, axes = plt.subplots(1, len(sample_indices), figsize=(4 * len(sample_indices), 5))
if len(sample_indices) == 1:
    axes = [axes]

DR_SEVERITY_COLORS = {
    0: '#2ecc71',  # Green — No DR
    1: '#f1c40f',  # Yellow — Mild
    2: '#e67e22',  # Orange — Moderate
    3: '#e74c3c',  # Red — Severe
    4: '#8e44ad'   # Purple — Proliferative
}

for ax, idx in zip(axes, sample_indices):
    img         = X_val[idx]               # Normalized [0,1]
    true_label  = y_val[idx]
    pred_probs  = final_model.predict(img[np.newaxis, ...], verbose=0)[0]
    pred_label  = np.argmax(pred_probs)
    confidence  = pred_probs[pred_label] * 100

    ax.imshow(img)
    ax.axis('off')

    is_correct = (pred_label == true_label)
    status     = '✅' if is_correct else '❌'
    color      = DR_SEVERITY_COLORS[pred_label]

    ax.set_title(
        f'{status}\nTrue : {CLASS_NAMES[true_label]}\n'
        f'Pred : {CLASS_NAMES[pred_label]}\n'
        f'Conf : {confidence:.1f}%',
        fontsize=9,
        color=color,
        fontweight='bold'
    )

fig.suptitle('Sample Predictions — One per DR Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n✅ Sanity check complete — predictions look good!')

## 📋 Step 17 — Final Summary Report

In [ ]:
# Print the complete project summary

print('=' * 70)
print('   🩺 DIABETIC RETINOPATHY AUTOML CLASSIFICATION — FINAL REPORT')
print('=' * 70)
print()
print('📐 INPUT CONFIGURATION')
print(f'   Image Resolution    : {IMG_SIZE[0]}×{IMG_SIZE[1]} px (STRICT 384×384)')
print(f'   Batch Size          : {BATCH_SIZE}')
print(f'   Number of Classes   : {NUM_CLASSES}')
print(f'   Class Names         : {CLASS_NAMES}')
print()
print('🤖 AUTOML SEARCH')
print(f'   Trials Attempted    : {MAX_TRIALS}')
print(f'   Epochs per Trial    : {AUTOML_EPOCHS} (STRICT: exactly 3)')
print(f'   AutoML Val Accuracy : {automl_acc * 100:.2f}%')
print()
print('🔧 FINE-TUNING')
print(f'   Layers Unfrozen     : ALL (full model)')
print(f'   Optimizer           : Adam')
print(f'   Learning Rate       : {FINETUNE_LR} (1e-5, STRICT)')
print(f'   Fine-tune Epochs    : {FINETUNE_EPOCHS} (STRICT: exactly 3)')
print(f'   Final Train Acc     : {finetune_history.history["accuracy"][-1]*100:.2f}%')
print(f'   Final Val Accuracy  : {finetune_history.history["val_accuracy"][-1]*100:.2f}%')
print()
print('📊 FINAL EVALUATION')
print(f'   Validation Accuracy : {final_accuracy * 100:.2f}%')
print(f'   Confusion Matrix    : /content/confusion_matrix.png')
print()
print('⚖️  CLASS IMBALANCE HANDLING')
for cls_id, weight in class_weights_dict.items():
    print(f'   Class {cls_id} weight       : {weight:.4f}')
print()
print('💾 MODEL ARTIFACTS')
print(f'   Drive Save Path     : {MODEL_SAVE_PATH}')
print(f'   Local Path          : /content/dr_model_384.keras')
print(f'   Format              : .keras (Keras v3)')
print()
print('🚀 STREAMLIT USAGE')
print('   import tensorflow as tf')
print('   model = tf.keras.models.load_model("dr_model_384.keras")')
print('   img   = tf.image.resize(img, (384, 384)) / 255.0')
print('   pred  = model.predict(img[tf.newaxis, ...])')
print('   class = tf.argmax(pred, axis=1).numpy()[0]')
print()
print('=' * 70)
print('✅ PROJECT COMPLETE — Model ready for Streamlit deployment!')
print('=' * 70)

## 🌐 Bonus — Streamlit UI Code Snippet

Save this as `app.py` and run `streamlit run app.py` with your `dr_model_384.keras` in the same folder.

In [ ]:
# This cell just PRINTS the Streamlit UI code — copy and save as app.py
# Do NOT run this in Colab — it is meant for local/Streamlit Cloud deployment

streamlit_code = '''
# ─── app.py — Diabetic Retinopathy Streamlit UI ────────────────────────────
# Run: streamlit run app.py
# Requires: streamlit, tensorflow, pillow, numpy

import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import io

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_PATH  = "dr_model_384.keras"   # Must be in same folder as app.py
IMG_SIZE    = (384, 384)
CLASS_NAMES = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]
CLASS_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

DR_INFO = {
    "No DR"            : "No signs of diabetic retinopathy detected.",
    "Mild DR"          : "Mild non-proliferative DR. Microaneurysms present.",
    "Moderate DR"      : "Moderate non-proliferative DR. More pronounced vascular changes.",
    "Severe DR"        : "Severe non-proliferative DR. Significant vascular damage.",
    "Proliferative DR" : "Proliferative DR. High-risk stage. Urgent medical attention required."
}

@st.cache_resource
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)

def preprocess_image(img: Image.Image) -> np.ndarray:
    img = img.convert("RGB")
    img = img.resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr[np.newaxis, ...]   # Add batch dimension

# ── UI Layout ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="DR Grading AI",
    page_icon="🩺",
    layout="centered"
)

st.title("🩺 Diabetic Retinopathy Grading System")
st.markdown("Upload a retinal fundus image to classify the DR severity level.")
st.warning("⚠️ For research use only. Not approved for clinical diagnosis.")

uploaded_file = st.file_uploader(
    "Upload Retinal Fundus Image",
    type=["jpg", "jpeg", "png"],
    help="Accepted formats: JPG, JPEG, PNG"
)

if uploaded_file is not None:
    # Display uploaded image
    img = Image.open(uploaded_file)
    col1, col2 = st.columns(2)
    with col1:
        st.image(img, caption="Uploaded Fundus Image", use_column_width=True)

    # Run prediction
    with st.spinner("🤖 Analyzing retinal image..."):
        model    = load_model()
        arr      = preprocess_image(img)
        preds    = model.predict(arr, verbose=0)[0]
        pred_cls = int(np.argmax(preds))
        confidence = float(preds[pred_cls]) * 100

    with col2:
        st.markdown(f"### Prediction: {CLASS_NAMES[pred_cls]}")
        st.markdown(f"**Confidence: {confidence:.1f}%**")
        st.markdown(DR_INFO[CLASS_NAMES[pred_cls]])

    # Probability bar chart
    st.markdown("### 📊 Class Probabilities")
    for i, (name, prob) in enumerate(zip(CLASS_NAMES, preds)):
        st.progress(float(prob), text=f"{name}: {prob*100:.1f}%")
'''

print(streamlit_code)

# Save the streamlit code as a file for download
with open('/content/app.py', 'w') as f:
    f.write(streamlit_code.strip())

print('\n✅ Streamlit app.py saved to /content/app.py')
print('   Download it along with dr_model_384.keras for deployment')

In [ ]:
# Download the Streamlit app.py file
from google.colab import files

print('📥 Downloading Streamlit app.py...')
files.download('/content/app.py')
print('✅ Done! You now have both dr_model_384.keras and app.py')
print('\nTo deploy locally:')
print('   pip install streamlit tensorflow pillow numpy')
print('   streamlit run app.py')